# PalanthAI — BGE-M3 Embedding

**1 clic = tout automatique** (embedding + upload Drive + notif n8n)


In [ ]:
# Drive mount + installation
from google.colab import drive
drive.mount('/content/drive', timeout_secs=300)
!pip install -q FlagEmbedding pyarrow tqdm pandas 2>&1 | tail -2
print('Setup OK')

In [ ]:
# Config
import pandas as pd, numpy as np, os, requests
from pathlib import Path

# Use Drive API (user OAuth) to read from shared folder
from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default()

from googleapiclient.discovery import build
drive_api = build('drive', 'v3', credentials=creds)

SHARED_FOLDER_ID = '13Mc5Nu2mgtQcsOFuN79Ug6-i1mRFPE9q'
OUTPUT_FOLDER_ID  = '13Mc5Nu2mgtQcsOFuN79Ug6-i1mRFPE9q'  # same folder for output
N8N_WEBHOOK = 'https://n8n.recall-agency.com/webhook/palanthai-colab-download'

# List CSV in shared folder via Drive API
results = drive_api.files().list(
    q=f"'{SHARED_FOLDER_ID}' in parents and name contains 'project_embeddings_20260331_5k_part0' and trashed=false",
    fields='files(id,name)').execute()
csv_files = results.get('files', [])
print(f'CSV trouve: {[f["name"] for f in csv_files]}')
CSV_FILES = [f['id'] for f in csv_files]  # list of file IDs


In [ ]:
# Load BGE-M3
import torch
print('GPU:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('Model OK')

In [ ]:
# Embed + save parquet
from tqdm.notebook import tqdm

for csv_path in CSV_FILES:
    df = pd.read_csv(csv_path)
    ids   = df['project_id'].tolist()
    texts = df['text'].fillna('').tolist()
    print(f'Embedding {len(texts)} projets...')
    out = model.encode(texts, batch_size=64, max_length=512,
                        return_dense=True, return_sparse=True)
    res = pd.DataFrame({
        'project_id': ids,
        'dense_vector': out['dense_vecs'].tolist(),
        'sparse_indices': [list(s.keys()) for s in out['lexical_weights']],
        'sparse_values':  [list(s.values()) for s in out['lexical_weights']],
    })
    for col in ['region']:
        if col in df.columns: res[col] = df[col]
    pq_name = csv_path.name.replace('.csv', '_vectors.parquet')
    local_pq = Path('/content/') / pq_name
    res.to_parquet(local_pq, index=False)
    print(f'Sauvegarde: {pq_name} ({local_pq.stat().st_size//1024}KB)')
    del df, res, out

print('Embedding TERMINE')

In [ ]:
# Upload parquet vers Drive (service account)
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)
drive_svc = build('drive', 'v3', credentials=creds)

for pq in sorted(Path('/content/').glob('*_vectors.parquet')):
    query = f"name='{pq.name}' and '{FOLDER_ID}' in parents and trashed=false"
    existing = drive_svc.files().list(q=query, fields='files(id)').execute()
    media = MediaFileUpload(str(pq), mimetype='application/octet-stream', resumable=True)
    if existing.get('files'):
        drive_svc.files().update(fileId=existing['files'][0]['id'],
            media_body=media, supportsAllDrives=True).execute()
        print(f'Updated: {pq.name}')
    else:
        drive_svc.files().create(
            body={'name': pq.name, 'parents': [FOLDER_ID]},
            media_body=media, fields='id', supportsAllDrives=True).execute()
        print(f'Uploaded: {pq.name}')

print('Upload Drive OK')

In [ ]:
# Notify n8n
pq_list = [p.name for p in sorted(Path('/content/').glob('*_vectors.parquet'))]
payload = {
    'status': 'parquet_uploaded',
    'parquet_files': pq_list,
    'collection': 'projects_v3',
    'count': len(pq_list),
    'source': 'colab'
}
try:
    r = requests.post(N8N_WEBHOOK, json=payload, timeout=30)
    print(f'n8n: {r.status_code} - {r.text[:200]}')
except Exception as e:
    print(f'n8n failed (check ingest service): {e}')

print('DONE — ', pq_list)